In [ ]:
def solve_facility_placement(nums, max_k):
    nums.sort()
    n = len(nums)
    
    # Precompute the cost of a single facility covering nums[i..j]
    cost = [[0] * n for _ in range(n)]
    for i in range(n):
        for j in range(i, n):
            median = nums[(i + j) // 2]
            total = 0
            for m in range(i, j + 1):
                total += abs(nums[m] - median)
            cost[i][j] = total
    
    # Initialize DP table and facility tracking
    dp = [[float('inf')] * (max_k + 1) for _ in range(n + 1)]
    dp[0][0] = 0
    facilities = [[[] for _ in range(max_k + 1)] for _ in range(n + 1)]
    
    for i in range(1, n + 1):
        for j in range(1, max_k + 1):
            for m in range(i):
                if dp[m][j - 1] + cost[m][i - 1] < dp[i][j]:
                    dp[i][j] = dp[m][j - 1] + cost[m][i - 1]
                    facilities[i][j] = facilities[m][j - 1] + [nums[(m + i - 1) // 2]]
    
    # Compute fragmentation percentage for each k
    fragmentation = []
    for k in range(1, max_k + 1):
        if dp[n][k] == float('inf'):
            fragmentation.append(0.0)
            continue
        sum_distances = dp[n][k]
        sum_facilities = sum(facilities[n][k])
        if sum_facilities == 0:
            fragmentation.append(0.0)
        else:
            fragmentation.append((sum_distances / sum_facilities) * 100)
    
    # Prepare results: for each k, return min_distance and facility_locations
    results = []
    for k in range(1, max_k + 1):
        results.append({
            'k': k,
            'min_distance': dp[n][k],
            'facility_locations': facilities[n][k],
            'fragmentation_percentage': fragmentation[k - 1]
        })
    
    return results

# Example usage:
# nums = [10, 20, 30, 40, 50]
# max_k = 3
# results = solve_facility_placement(nums, max_k)
# for result in results:
#     print(f"k={result['k']}, min_distance={result['min_distance']}, facilities={result['facility_locations']}, fragmentation={result['fragmentation_percentage']}%")

In [2]:
import pandas as pd
df = pd.read_csv('/proj/latencymodel-PG0/hongshu/traces/meta2024_50m.csv')

In [ ]:
import csv

def read_and_adjust_object_sizes(csv_file_path):
    nums = []
    with open(csv_file_path, mode='r') as file:
        reader = csv.DictReader(file)
        for row in reader:
            object_id = row['object_id']
            object_size = int(row['object_size'])
            object_size = max(24, object_size)
            object_size += 32 + len(str(object_id))
            nums.append(object_size)
    return nums



In [4]:
# Example usage:
csv_file_path = '/proj/latencymodel-PG0/hongshu/traces/meta2024_50m.csv'
nums = read_and_adjust_object_sizes(csv_file_path)

In [ ]:
results = solve_facility_placement(nums, 100)